# TP1 — Risque de Bayes, sur-apprentissage, évaluation
**CNFSDP — Introduction au Machine Learning** · L3 S5 · 2026-2027 · Rois Céti DIMBAMBA · chapitre 1 · 30 min en salle

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rdimbamba/ml-cnfsdp/blob/main/notebooks/TP1.ipynb)

Partie A : régression, sur-apprentissage d'un arbre et variance. Partie B : classification, erreur de Bayes, intervalle de confiance, validation croisée, ROC.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve, roc_auc_score
VERT, OR, GRIS = "#126B2A", "#B8860B", "#8A8A8A"   # couleurs du cours

## Partie A — Régression : $Y=\sin(2\pi X)+\varepsilon$, $X\sim\mathcal U[0,1]$, $\sigma=0{,}3$
Ici le meilleur prédicteur est $f^*(x)=\sin(2\pi x)$ et l'erreur irréductible vaut $R^*=\sigma^2=0{,}09$ (modèle signal + bruit du cours).

In [ ]:
rng = np.random.default_rng(1)
sigma = 0.3
def simuler_reg(n):
    X = rng.uniform(0, 1, (n, 1))
    return X, np.sin(2 * np.pi * X[:, 0]) + sigma * rng.standard_normal(n)

X, y = simuler_reg(200)        # apprentissage
Xt, yt = simuler_reg(5_000)    # test

### Question 1 — Risques d'apprentissage et test selon la profondeur
Pour chaque profondeur $p=1,\dots,15$, ajuster `DecisionTreeRegressor(max_depth=p)` sur $(X,y)$, calculer l'erreur quadratique moyenne sur l'apprentissage et sur le test, puis tracer les deux courbes avec la droite $R^*=\sigma^2$.

In [ ]:
prof = range(1, 16)
err_app, err_test = [], []
# À compléter : boucle sur les profondeurs, ajustement, calcul des deux erreurs
# puis tracé des deux courbes et de la droite R* = sigma**2


**À commenter :** pourquoi l'erreur d'apprentissage tend-elle vers 0 ? Pourquoi le risque test ne descend-il jamais sous $\sigma^2$ ?

### Question 2 — Variance d'un arbre profond
Tirer 20 échantillons bootstrap de $(X,y)$ (tirage avec remise de 200 indices). Sur chacun, ajuster un arbre de profondeur 15 et un arbre de profondeur 3, et tracer leurs prédictions sur une grille de $[0,1]$. Estimer la variance moyenne des prédictions dans les deux cas.

In [ ]:
grille = np.linspace(0, 1, 400)[:, None]
pred = {3: [], 15: []}
# À compléter : 20 tirages bootstrap, ajustement des deux arbres, prédictions sur la grille,
# tracés, puis variance moyenne des prédictions (np.array(pred[p]).var(axis=0).mean())


## Partie B — Classification : mélange gaussien du cours
$\P(Y=1)=\frac12$, $X\mid Y=k\sim\mathcal N_2(1{,}5\,k\,(1,1), I_2)$. Le classifieur de Bayes prédit 1 si $x_1+x_2>1{,}5$, et son erreur vaut $R^*=\Phi(-1{,}5/\sqrt2)\approx0{,}144$.

In [ ]:
rng = np.random.default_rng(0)
def simuler_cl(n):
    y = rng.integers(0, 2, n)
    return rng.standard_normal((n, 2)) + 1.5 * y[:, None], y

X, y = simuler_cl(1_000)
Xt, yt = simuler_cl(10_000)
R_star = norm.cdf(-1.5 / np.sqrt(2))

### Questions 3 et 4 — Taux d'erreur test et intervalles de confiance
Comparer les taux d'erreur test de la régression logistique, d'un arbre de profondeur illimitée et du classifieur de Bayes à $R^*\approx0{,}144$. Donner pour chacun l'intervalle de confiance à 95 % : $\hat p\pm1{,}96\sqrt{\hat p(1-\hat p)/m}$.

In [ ]:
m = len(yt)
# À compléter : taux d'erreur test de LogisticRegression(), DecisionTreeClassifier() et du classifieur de Bayes
# (prédit 1 si x1 + x2 > 1,5), avec l'intervalle p ± 1,96 * sqrt(p (1 - p) / m)


### Question 5 — Validation croisée et courbe ROC
Choisir la profondeur d'un arbre parmi $1,\dots,10$ par validation croisée à 5 blocs sur $(X,y)$, réentraîner sur tout $(X,y)$, puis tracer la courbe ROC sur le test et calculer l'AUC. Vérifier que l'AUC coïncide avec la statistique de Mann-Whitney.

In [ ]:
# À compléter : cross_val_score pour max_depth = 1..10, choix de la meilleure profondeur,
# réentraînement, roc_curve et roc_auc_score sur le test, comparaison avec Mann-Whitney


**Indice.** Un arbre produit beaucoup de scores égaux : dans la statistique de Mann-Whitney, compter les ex aequo pour moitié.